# Regressão linear com PIB e Índice ABCR

**Checkpoint 02 – MLAM 2026/2 – FIAP**

**Integrantes:** Renan Eskildssen – RM 571097 *(adicione aqui os demais integrantes, se o trabalho for em grupo)*

**Pergunta de pesquisa:** existe relação entre a atividade econômica brasileira (índice de volume do PIB) e o fluxo de veículos nas rodovias pedagiadas (Índice ABCR)? Um modelo de regressão linear consegue estimar o Índice ABCR a partir do PIB?

**Roteiro do notebook**
1. Fontes consultadas
2. Configuração
3. Coleta do PIB (IBGE/SIDRA)
4. Coleta do Índice ABCR
5. Montagem da base anual (2006–2025)
6. Análise da relação (dispersão e correlação)
7. Treinamento do modelo
8. Avaliação (MAE, MSE, R²)
9. Interpretação e conclusão
10. Dificuldades encontradas e soluções

## 1. Fontes consultadas

| Fonte | Série | Uso |
|---|---|---|
| IBGE – SIDRA, [Tabela 1620](https://sidra.ibge.gov.br/tabela/1620) | Série encadeada do índice de volume trimestral do PIB (base: média de 1995 = 100), setor "PIB a preços de mercado", **sem ajuste sazonal** | Variável de entrada `X` (`PIB_indice`) |
| ABCR – [Índice ABCR](https://melhoresrodovias.org.br/indice-abcr_2/) (botão "Ver histórico") | Índice de fluxo **total** de veículos, Brasil, **série original (sem ajuste sazonal)**, mensal | Variável a estimar `y` (`ABCR_indice`) |

**Decisões metodológicas (justificativas)**
- **Sem ajuste sazonal:** é o que o enunciado pede. Além disso, ao agregar por ano (média de 4 trimestres e de 12 meses) a sazonalidade se anula, então o ano completo é comparável.
- **Números-índice, não variações %:** o enunciado pede os níveis. Variações percentuais descartariam a tendência de longo prazo que é justamente o que queremos relacionar.
- **Período 2006–2025 (20 anos completos):** só entram anos com os 4 trimestres do PIB **e** os 12 meses da ABCR, para que as médias anuais não sejam distorcidas por anos incompletos.
- **Divisão cronológica (16 treino / 4 teste):** é uma série temporal, então embaralhar vazaria informação do futuro para o treino.

## 2. Configuração

In [ ]:
from pathlib import Path
import glob, re, warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ANO_INICIO, ANO_FIM = 2006, 2025      # 20 anos completos
N_TREINO = 16                          # primeiros 16 anos = treino; últimos 4 = teste

DADOS = Path("dados");           DADOS.mkdir(exist_ok=True)
RESULTADOS = Path("resultados"); RESULTADOS.mkdir(exist_ok=True)

# API pública do SIDRA: tabela 1620, Brasil (n1), todos os trimestres, todos os setores
# (filtramos "PIB a preços de mercado" no código, o que evita depender de códigos de categoria)
URL_SIDRA = "https://apisidra.ibge.gov.br/values/t/1620/n1/all/v/all/p/all/c11255/all"

# Arquivo "Ver histórico" da página do Índice ABCR. O nome muda a cada divulgação mensal
# (abcr_MMAA.xlsx). Se esse link falhar, baixe o arquivo manualmente pela página da ABCR e
# coloque na pasta dados/ (qualquer nome abcr*.xlsx).
URL_ABCR_XLSX = "https://melhoresrodovias.org.br/wp-content/uploads/2026/09/abcr_0826.xlsx"

HEADERS = {"User-Agent": "Mozilla/5.0 (trabalho academico FIAP)"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})

## 3. Coleta do PIB (IBGE/SIDRA)

A página da Tabela 1620 é montada por JavaScript, então não dá para copiar os dados da tela. A solução foi usar a **API do SIDRA**, que devolve a tabela em JSON. O código abaixo baixa a série trimestral, mantém apenas "PIB a preços de mercado" e salva uma cópia em `dados/pib_trimestral.csv` (se o arquivo já existir, ele é reaproveitado, o que deixa o notebook reprodutível mesmo sem internet).

In [ ]:
def baixar_pib_sidra():
    resp = requests.get(URL_SIDRA, headers=HEADERS, timeout=90)
    resp.raise_for_status()
    dados = resp.json()
    cab, linhas = dados[0], dados[1:]          # a 1ª linha do JSON é o cabeçalho
    df = pd.DataFrame(linhas)

    # descobre, pelo cabeçalho, quais chaves guardam trimestre e setor
    k_trim = next(k for k, v in cab.items() if v.strip().lower() == "trimestre")
    k_setor = next(k for k, v in cab.items() if "setores" in v.lower() and k.endswith("N"))
    k_trim_cod = k_trim[:-1] + "C"

    pib = df[df[k_setor].str.strip() == "PIB a preços de mercado"].copy()
    if pib.empty:
        raise ValueError(f"Setor não encontrado. Setores disponíveis: {df[k_setor].unique()}")

    pib["Ano"] = pib[k_trim_cod].str[:4].astype(int)
    pib["Trimestre"] = pib[k_trim_cod].str[4:].astype(int)
    pib["PIB_indice"] = pd.to_numeric(pib["V"], errors="coerce")
    return pib[["Ano", "Trimestre", "PIB_indice"]].sort_values(["Ano", "Trimestre"]).reset_index(drop=True)


arq_pib = DADOS / "pib_trimestral.csv"
if arq_pib.exists():
    pib_tri = pd.read_csv(arq_pib)
    print("PIB lido de", arq_pib)
else:
    pib_tri = baixar_pib_sidra()
    pib_tri.to_csv(arq_pib, index=False)
    print("PIB baixado do SIDRA e salvo em", arq_pib)

print(pib_tri.shape)
pib_tri.tail(8)

## 4. Coleta do Índice ABCR

O histórico da ABCR é uma planilha Excel com várias séries (Brasil / São Paulo / Rio de Janeiro; leves / pesados / total; original / dessazonalizada). Como o enunciado pede **Brasil, fluxo total, sem ajuste sazonal**, o código varre as abas, pontua cada coluna pelo texto do cabeçalho e escolhe a mais adequada. **O resultado da escolha é impresso para conferência** e, se necessário, é possível forçar a aba/coluna nas variáveis `ABA_FORCADA` e `COL_FORCADA`.

In [ ]:
ABA_FORCADA, COL_FORCADA = None, None   # use se quiser forçar a escolha (nome da aba / índice da coluna)

def _txt(x):
    return "" if pd.isna(x) else str(x).strip().lower()

def _pontuar(txt):
    s = 0
    if "total" in txt: s += 3
    if "brasil" in txt: s += 2
    if re.search(r"dessaz|com ajuste|ajustad|\bs\.?a\.?\b", txt): s -= 10
    if re.search(r"sem ajuste|original|bruto|observad", txt): s += 3
    if re.search(r"leve|pesad|s[aã]o paulo|rio de janeiro|\bsp\b|\brj\b", txt): s -= 4
    return s

def achar_serie_abcr(abas):
    """Varre todas as abas e devolve (serie_mensal, descricao_da_escolha, tabela_de_candidatos)."""
    candidatos = []
    for nome_aba, bruto in abas.items():
        bruto = bruto.dropna(how="all").dropna(axis=1, how="all").reset_index(drop=True)
        if len(bruto) < 50:
            continue
        # 1) coluna de datas: a que tiver mais valores interpretáveis como data entre 2000 e 2035
        melhor_col, melhor_dt, melhor_n = None, None, 0
        for c in bruto.columns:
            dt = pd.to_datetime(bruto[c], errors="coerce")
            ok = dt.dt.year.between(2000, 2035)
            if ok.sum() > melhor_n:
                melhor_col, melhor_dt, melhor_n = c, dt.where(ok), int(ok.sum())
        if melhor_n < 100:
            continue
        primeira = melhor_dt.first_valid_index()
        cab = bruto.iloc[:primeira].ffill(axis=1)           # cabeçalho (células mescladas se repetem à direita)
        for c in bruto.columns:
            if c == melhor_col:
                continue
            valores = pd.to_numeric(bruto.loc[melhor_dt.notna(), c], errors="coerce")
            if valores.notna().mean() < 0.9:
                continue
            texto = " ".join([_txt(nome_aba)] + [_txt(v) for v in cab[c].tolist()])
            serie = pd.Series(valores.values, index=melhor_dt[melhor_dt.notna()].dt.to_period("M"))
            candidatos.append({"aba": nome_aba, "coluna": c, "cabecalho": texto.strip(),
                               "pontos": _pontuar(texto), "n_meses": int(serie.notna().sum()), "serie": serie})
    if not candidatos:
        raise ValueError("Nenhuma série mensal encontrada. Veja a seção de dificuldades: use o fallback dados/abcr_mensal.csv.")
    tabela = pd.DataFrame(candidatos).drop(columns="serie").sort_values("pontos", ascending=False)

    if ABA_FORCADA is not None:
        escolhido = next(c for c in candidatos if c["aba"] == ABA_FORCADA and c["coluna"] == COL_FORCADA)
    else:
        escolhido = max(candidatos, key=lambda c: (c["pontos"], c["n_meses"]))
        if escolhido["pontos"] < 1:
            raise ValueError("Não consegui identificar com segurança a série 'Brasil / total / original'. "
                             "Veja a tabela de candidatos e force a escolha com ABA_FORCADA/COL_FORCADA.")
    return escolhido["serie"], f"aba={escolhido['aba']!r}, coluna={escolhido['coluna']}, cabeçalho={escolhido['cabecalho']!r}", tabela


def obter_abcr_mensal():
    arq_csv = DADOS / "abcr_mensal.csv"
    if arq_csv.exists():
        print("ABCR lido de", arq_csv)
        return pd.read_csv(arq_csv), None, None

    arquivos = sorted(glob.glob(str(DADOS / "abcr*.xlsx")))
    if arquivos:
        xlsx = Path(arquivos[-1]); print("ABCR: usando arquivo local", xlsx)
    else:
        xlsx = DADOS / Path(URL_ABCR_XLSX).name
        resp = requests.get(URL_ABCR_XLSX, headers=HEADERS, timeout=90); resp.raise_for_status()
        xlsx.write_bytes(resp.content); print("ABCR baixado e salvo em", xlsx)

    abas = pd.read_excel(xlsx, sheet_name=None, header=None)
    serie, escolha, tabela = achar_serie_abcr(abas)
    mensal = serie.rename("ABCR_indice").rename_axis("Periodo").reset_index()
    mensal["Ano"] = mensal["Periodo"].dt.year
    mensal["Mes"] = mensal["Periodo"].dt.month
    mensal = mensal[["Ano", "Mes", "ABCR_indice"]].dropna().drop_duplicates(["Ano", "Mes"]).sort_values(["Ano", "Mes"])
    mensal.to_csv(arq_csv, index=False)
    return mensal.reset_index(drop=True), escolha, tabela


abcr_mensal, escolha, candidatos = obter_abcr_mensal()
if escolha:
    print("\nSérie escolhida ->", escolha)
    display(candidatos.head(6))
print(abcr_mensal.shape)
abcr_mensal.tail(6)

## 5. Montagem da base anual

- `PIB_indice` = média dos **4** índices trimestrais do ano;
- `ABCR_indice` = média dos **12** índices mensais do ano;
- só entram anos **completos** nas duas séries, e cada linha traz os dois indicadores do mesmo ano.

In [ ]:
pib_anual = (pib_tri.dropna(subset=["PIB_indice"]).groupby("Ano")["PIB_indice"]
             .agg(PIB_indice="mean", n_tri="count"))
abcr_anual = (abcr_mensal.dropna(subset=["ABCR_indice"]).groupby("Ano")["ABCR_indice"]
              .agg(ABCR_indice="mean", n_mes="count"))

base = pib_anual.join(abcr_anual, how="inner").reset_index()
base = base[base["Ano"].between(ANO_INICIO, ANO_FIM)]

# conferência de completude
incompletos = base[(base["n_tri"] != 4) | (base["n_mes"] != 12)]
assert incompletos.empty, f"Anos incompletos:\n{incompletos}"
assert len(base) == ANO_FIM - ANO_INICIO + 1, f"Esperava {ANO_FIM-ANO_INICIO+1} anos, vieram {len(base)}: {base['Ano'].tolist()}"

df = base[["Ano", "PIB_indice", "ABCR_indice"]].round(4).reset_index(drop=True)
df.to_csv(DADOS / "base_anual.csv", index=False)
print("Base anual salva em dados/base_anual.csv")
df

In [ ]:
# Inspeção rápida da base
print(df.shape); print(); df.info(); print()
print("Valores nulos:\n", df.isnull().sum(), "\n")
df.describe().round(2)

## 6. Análise da relação

In [ ]:
r_pearson = df["PIB_indice"].corr(df["ABCR_indice"])
r_spearman = df["PIB_indice"].corr(df["ABCR_indice"], method="spearman")
print(f"Correlação de Pearson : {r_pearson:.4f}")
print(f"Correlação de Spearman: {r_spearman:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df["PIB_indice"], df["ABCR_indice"], s=45, color="tab:blue")
for _, l in df.iterrows():
    ax.annotate(int(l["Ano"]), (l["PIB_indice"], l["ABCR_indice"]), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set_xlabel("Índice de volume do PIB (média 1995 = 100)")
ax.set_ylabel("Índice ABCR (fluxo total de veículos)")
ax.set_title(f"PIB x Índice ABCR, {ANO_INICIO}–{ANO_FIM} (r = {r_pearson:.3f})")
plt.tight_layout(); plt.savefig(RESULTADOS / "dispersao.png"); plt.show()

# as duas séries ao longo do tempo
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(df["Ano"], df["PIB_indice"], marker="o", color="tab:blue", label="PIB")
ax1.set_ylabel("PIB (índice)", color="tab:blue")
ax2 = ax1.twinx(); ax2.grid(False)
ax2.plot(df["Ano"], df["ABCR_indice"], marker="s", color="tab:orange", label="ABCR")
ax2.set_ylabel("ABCR (índice)", color="tab:orange")
ax1.set_title("Evolução anual das duas séries")
plt.tight_layout(); plt.savefig(RESULTADOS / "series_tempo.png"); plt.show()

## 7. Treinamento do modelo

`X = PIB_indice`, `y = ABCR_indice`. **Treino:** 16 primeiros anos; **teste:** 4 últimos anos, mantendo a ordem cronológica (sem embaralhar).

In [ ]:
X = df[["PIB_indice"]]
y = df["ABCR_indice"]

X_treino, X_teste = X.iloc[:N_TREINO], X.iloc[N_TREINO:]
y_treino, y_teste = y.iloc[:N_TREINO], y.iloc[N_TREINO:]
print("Treino:", df["Ano"].iloc[0], "a", df["Ano"].iloc[N_TREINO-1], f"({len(X_treino)} anos)")
print("Teste :", df["Ano"].iloc[N_TREINO], "a", df["Ano"].iloc[-1], f"({len(X_teste)} anos)")

modelo = LinearRegression()
modelo.fit(X_treino, y_treino)
y_pred = modelo.predict(X_teste)

print(f"\nABCR_indice = {modelo.intercept_:.3f} + {modelo.coef_[0]:.3f} x PIB_indice")

## 8. Avaliação dos resultados (conjunto de teste)

In [ ]:
mae = mean_absolute_error(y_teste, y_pred)
mse = mean_squared_error(y_teste, y_pred)
r2 = r2_score(y_teste, y_pred)

metricas = pd.DataFrame({"Métrica": ["MAE", "MSE", "R²"], "Valor": [mae, mse, r2]})
metricas.to_csv(RESULTADOS / "metricas.csv", index=False)
display(metricas.round(4))

comparacao = pd.DataFrame({
    "Ano": df["Ano"].iloc[N_TREINO:].values,
    "PIB_indice": X_teste["PIB_indice"].values,
    "ABCR_observado": y_teste.values,
    "ABCR_previsto": y_pred,
})
comparacao["Erro"] = comparacao["ABCR_previsto"] - comparacao["ABCR_observado"]
comparacao["Erro_%"] = 100 * comparacao["Erro"] / comparacao["ABCR_observado"]
comparacao.round(3).to_csv(RESULTADOS / "previsoes_teste.csv", index=False)
display(comparacao.round(3))

# gráficos: reta ajustada e observado x previsto
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(X_treino, y_treino, label="Treino", color="tab:blue")
ax[0].scatter(X_teste, y_teste, label="Teste (observado)", color="tab:orange")
ax[0].scatter(X_teste, y_pred, label="Teste (previsto)", marker="x", s=70, color="tab:red")
xs = np.linspace(X["PIB_indice"].min(), X["PIB_indice"].max(), 50).reshape(-1, 1)
ax[0].plot(xs, modelo.predict(pd.DataFrame(xs, columns=["PIB_indice"])), color="gray", ls="--", label="Reta ajustada")
ax[0].set_xlabel("PIB_indice"); ax[0].set_ylabel("ABCR_indice"); ax[0].legend(); ax[0].set_title("Modelo ajustado")
ax[1].plot(comparacao["Ano"], comparacao["ABCR_observado"], marker="o", label="Observado")
ax[1].plot(comparacao["Ano"], comparacao["ABCR_previsto"], marker="x", label="Previsto")
ax[1].set_xticks(comparacao["Ano"]); ax[1].legend(); ax[1].set_title("Teste: observado x previsto")
plt.tight_layout(); plt.savefig(RESULTADOS / "modelo_teste.png"); plt.show()

### Como ler as métricas
- **MAE (erro absoluto médio):** média do tamanho dos erros, na mesma unidade do índice ABCR. Ex.: MAE = 5 significa que, em média, a previsão erra 5 pontos de índice.
- **MSE (erro quadrático médio):** média dos erros ao quadrado; penaliza mais os erros grandes e fica em "pontos²" (por isso é mais difícil de interpretar diretamente que o MAE).
- **R²:** quanto da variação do ABCR no teste o modelo explica em comparação com simplesmente prever a média. R² = 1 é perfeito; R² = 0 equivale a prever a média; **R² negativo significa pior que prever a média**. Como o conjunto de teste tem só 4 pontos, o R² é muito sensível e deve ser lido com cautela.

## 9. Interpretação e conclusão

In [ ]:
def forca(r):
    a = abs(r)
    return "muito forte" if a >= .9 else "forte" if a >= .7 else "moderada" if a >= .4 else "fraca"

media_teste = y_teste.mean()
mae_pct = 100 * mae / media_teste
mae_naive = mean_absolute_error(y_teste, np.full(len(y_teste), y_treino.mean()))
fora = int((X_teste["PIB_indice"] > X_treino["PIB_indice"].max()).sum())

print(f"1) Relação: correlação {forca(r_pearson)} e {'positiva' if r_pearson > 0 else 'negativa'} (Pearson = {r_pearson:.3f}; Spearman = {r_spearman:.3f}).")
print(f"2) Reta estimada: ABCR = {modelo.intercept_:.2f} + {modelo.coef_[0]:.2f} x PIB. "
      f"Cada ponto a mais no índice do PIB está associado a {modelo.coef_[0]:.2f} ponto(s) a mais no índice ABCR.")
print(f"3) Desempenho no teste ({df['Ano'].iloc[N_TREINO]}–{df['Ano'].iloc[-1]}): MAE = {mae:.2f} (≈ {mae_pct:.1f}% da média observada de {media_teste:.1f}), "
      f"MSE = {mse:.2f}, R² = {r2:.3f}.")
print(f"4) Referência: prever sempre a média do treino teria MAE = {mae_naive:.2f}; o modelo "
      f"{'melhora' if mae < mae_naive else 'NÃO melhora'} o MAE em relação a essa referência.")
if r2 < 0:
    print("5) R² negativo: no período de teste o modelo foi pior do que prever a média; a relação aprendida no treino não se manteve bem.")
elif r2 < .7:
    print("5) R² moderado: o modelo captura parte da variação, mas não toda.")
else:
    print("5) R² alto: o modelo acompanha bem a variação do ABCR no período de teste.")
if fora:
    print(f"6) Atenção: {fora} dos {len(X_teste)} anos de teste têm PIB acima do maior valor visto no treino; "
          "o modelo está extrapolando, o que costuma piorar as previsões.")
print("7) Correlação não é causalidade: PIB e fluxo de veículos podem se mover juntos por influência de fatores comuns "
      "(consumo, produção agrícola e industrial, tendência de longo prazo, mudanças nas praças de pedágio incluídas no índice), "
      "e duas séries com tendência crescente tendem a apresentar correlação alta mesmo sem relação direta.")

### Conclusão

*(Revise esta seção com as suas palavras depois de executar o notebook, usando os números impressos acima.)*

- O gráfico de dispersão e a correlação indicam o **grau e o sentido** da relação entre PIB e fluxo de veículos pedagiados no período 2006–2025.
- O modelo de regressão linear simples (apenas o PIB como entrada) foi avaliado em 4 anos nunca vistos no treino; **MAE, MSE e R²** resumem o tamanho do erro e a capacidade de explicar a variação do ABCR.
- Limitações: (i) apenas 20 observações anuais e só 4 no teste; (ii) as duas séries têm tendência no longo prazo, o que facilita a correlação alta; (iii) o Índice ABCR cobre somente rodovias **concedidas** e passa por revisões de base; (iv) eventos extraordinários (crise de 2015–2016, pandemia em 2020) estão no treino e podem distorcer a reta; (v) uma regressão simples não separa causa e efeito.
- Possíveis melhorias: usar variações percentuais anuais, séries mensais/trimestrais (mais pontos), incluir outras variáveis (indústria, preço de combustível) ou validação cruzada temporal (`TimeSeriesSplit`).

## 10. Dificuldades encontradas e soluções adotadas

| Dificuldade | Solução |
|---|---|
| A página da Tabela 1620 do SIDRA é carregada por JavaScript, então não há tabela para copiar/colar. | Usar a **API pública do SIDRA** (`apisidra.ibge.gov.br`), que devolve a tabela em JSON, e filtrar "PIB a preços de mercado" no código. |
| O SIDRA devolve vários setores e uma linha de cabeçalho misturada aos dados. | Ler o cabeçalho do JSON para descobrir quais campos são trimestre e setor, em vez de fixar nomes de colunas. |
| O histórico da ABCR é um `.xlsx` com muitas séries (Brasil/SP/RJ, leves/pesados/total, original/dessazonalizada). | Função que pontua o texto dos cabeçalhos, escolhe "Brasil, total, original" e **imprime a escolha** para conferência, com opção de forçar a aba/coluna. |
| O nome do arquivo da ABCR muda a cada mês (`abcr_MMAA.xlsx`). | O notebook aceita qualquer `dados/abcr*.xlsx` e salva a série mensal em `dados/abcr_mensal.csv`. |
| Médias anuais seriam enganosas com anos incompletos. | `assert` que exige 4 trimestres e 12 meses por ano e 20 anos no total. |
| Séries temporais não podem ser embaralhadas. | Divisão cronológica manual (`iloc`), sem `train_test_split` aleatório. |

*(Acrescente aqui qualquer outra dificuldade que você encontrar ao rodar.)*

## Bloco para colar no README (resultados)

In [ ]:
bloco = f"""
| Métrica (teste {df['Ano'].iloc[N_TREINO]}–{df['Ano'].iloc[-1]}) | Valor |
|---|---|
| MAE | {mae:.3f} |
| MSE | {mse:.3f} |
| R²  | {r2:.4f} |

Correlação de Pearson entre PIB_indice e ABCR_indice: **{r_pearson:.3f}**
Reta ajustada: `ABCR_indice = {modelo.intercept_:.3f} + {modelo.coef_[0]:.3f} x PIB_indice`
"""
print(bloco)
(RESULTADOS / "bloco_readme.md").write_text(bloco, encoding="utf-8")